# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all available record sets
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"@id: {record_set['@id']} | Name: {record_set.get('name', 'N/A')}")

# Display fields and columns for each record set
for record_set in metadata.record_sets:
    print(f"\nRecordSet: {record_set['@id']}")
    print('  Fields:')
    for field in record_set.get('fields', []):
        print(f"    - @id: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
    if 'columns' in record_set:
        print('  Columns:')
        for column in record_set['columns']:
            print(f"    - @id: {column['@id']} | Name: {column.get('name', 'N/A')} | DataType: {column.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List record set @ids for the dataset
record_sets_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    # Each record set's @id will be referenced explicitly
    print(f"Loading data for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields (@id) for {record_set_id}: {df.columns.tolist()}")
    else:
        print(f"No records found for record set {record_set_id}.")

# Example: Show first 5 records for the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nSample from RecordSet `{first_rs_id}`:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping, referencing fields by their `@id`.

In [ ]:
# Select a record set and numeric field for analysis
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Use first record set available
    df = dataframes[record_set_id]
    # Try to automatically find a numeric field by inspecting dtypes
    candidate_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not candidate_numeric_fields:
        # Try converting columns to numeric, suppress errors
        numeric_candidates = []
        for c in df.columns:
            try:
                df[c + "_float"] = pd.to_numeric(df[c], errors='coerce')
                if df[c + "_float"].notnull().any():
                    numeric_candidates.append(c + "_float")
            except:
                pass
        candidate_numeric_fields = numeric_candidates
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Remove NaN for analysis
        df_clean = df.dropna(subset=[numeric_field])
        # Filter: numeric_field > threshold (use median as threshold for demo)
        threshold = df_clean[numeric_field].median() if len(df_clean) > 0 else 1
        filtered_df = df_clean[df_clean[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Group by a likely categorical field
        group_field = None
        # Try finding a non-numeric, non-unique field
        for c in df.columns:
            if c != numeric_field and df[c].nunique() < len(df) // 2:
                group_field = c
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found; skipping group-by step.')
    else:
        print("No numeric field detected for EDA.")
else:
    print("No loaded dataframes to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields. We'll demonstrate a histogram for the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} in RecordSet: {record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field available, or data not loaded, for visualization.")

## 6. Conclusion
We explored the FAIR^2 Croissant dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Using `mlcroissant`, we:

- Discovered available record sets and fields using the dataset's `@id`s.
- Loaded data directly into DataFrames for programmatic exploration.
- Filtered, normalized, grouped, and visualized the data using field `@id` references.

Further analysis may include deeper comparisons, feature engineering, or domain-relevant statistical analyses based on research questions.